# C 1s peak-parameter variational encoder–decoder

**Input:** the noisy C 1s count vector from the generated NeXus (`.nxs`) file. **Output:** six components' presence, lmfit profile family, center, FWHM, model sigma, shape-specific parameters, height, integrated area, and area fraction. The latent vector is trained only through these peak-data targets and a small KL regularizer. There is **no spectrum decoder or spectrum reconstruction loss**. This is a supervised variational encoder–decoder with a VAE-style latent space, rather than a conventional input-reconstructing VAE.

The labels are the generator's ground truth, not parameters fitted back from noisy counts. The model supplies candidate parameters; the final cell demonstrates an lmfit fit initialized from those predictions. Splits are by generated case, so repeated acquisitions cannot cross splits. Run `c1s_lmfit_dataset.py` separately before running this notebook; this notebook never invokes the generator.

In [ ]:
from pathlib import Path
import csv
import json
import random

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path.cwd()
NXS_PATH = PROJECT_DIR / "datasets" / "c1s_nexus" / "c1s_atmospheric_v2.nxs"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "c1s_nexus_peak_vae"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "peak_parameter_vae.pt"

LATENT_DIM = 24
BATCH_SIZE = 256
EPOCHS = 150
PATIENCE = 20
LEARNING_RATE = 3e-4
BETA_KL = 1e-3
PRESENCE_THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Read the NeXus spectra and peak labels

Only `/entry/data/counts` is used as model input. The clean expected spectrum, background, and native acquisitions are not training targets. Shape codes are `0=absent`, `1=Voigt`, `2=pseudo-Voigt`, `3=Doniach Gaussian`, `4=Gaussian`.

In [ ]:
if not NXS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {NXS_PATH}. Run c1s_lmfit_dataset.py separately, then rerun this notebook."
    )

PEAK_FIELDS = (
    "peak_present", "peak_shape_code", "peak_center_ev", "peak_fwhm_ev",
    "peak_model_sigma_ev", "peak_lorentz_gamma_ev",
    "peak_pseudo_voigt_fraction", "peak_asymmetry_gamma",
    "peak_height_counts", "peak_area_counts_ev", "peak_area_fraction",
)
SCAN_FIELDS = ("case_id", "split", "family", "background_kind")
with h5py.File(NXS_PATH, "r") as nexus:
    spectra = np.asarray(nexus["entry/data/counts"], dtype=np.float32)
    energy_ev = np.asarray(nexus["entry/data/binding_energy"], dtype=np.float32)
    labels_group = nexus["entry/labels"]
    component_names = tuple(labels_group["component_names"].asstr()[:])
    shape_names = tuple(labels_group["shape_names"].asstr()[:])
    labels = {name: np.asarray(labels_group[name]) for name in PEAK_FIELDS}
    labels["case_id"] = np.asarray(labels_group["case_id"], dtype=np.int64)
    for name in ("split", "family", "background_kind"):
        labels[name] = np.asarray(labels_group[name].asstr()[:])
    script_sha256 = nexus["entry/simulation/script_sha256"].asstr()[()]

n_scans, n_points = spectra.shape
n_components = len(component_names)
assert n_scans > 0 and n_points == len(energy_ev)
assert n_components == 6, f"Expected six components, found {n_components}"
assert len(shape_names) == 5 and shape_names[0] == "absent"
assert np.all(np.isfinite(spectra)) and np.all(spectra >= 0)
assert np.all(np.diff(energy_ev) > 0), "Expected an increasing common binding-energy grid"
for name in PEAK_FIELDS:
    assert labels[name].shape == (n_scans, n_components), (name, labels[name].shape)
for name in SCAN_FIELDS:
    assert len(labels[name]) == n_scans, name
present = labels["peak_present"].astype(bool)
shape_code = labels["peak_shape_code"].astype(np.int64)
assert np.all((shape_code >= 0) & (shape_code < len(shape_names)))
assert np.array_equal(shape_code > 0, present)
for name in ("peak_center_ev", "peak_fwhm_ev", "peak_model_sigma_ev",
             "peak_height_counts", "peak_area_counts_ev", "peak_area_fraction"):
    assert np.all(np.isfinite(labels[name][present])), f"Missing {name} for a present peak"
assert np.allclose(labels["peak_area_fraction"].sum(axis=1), 1.0, atol=1e-4)

manifest = {
    "path": str(NXS_PATH.resolve()),
    "bytes": NXS_PATH.stat().st_size,
    "mtime_ns": NXS_PATH.stat().st_mtime_ns,
    "generator_sha256": script_sha256,
    "scans": n_scans,
    "energy_points": n_points,
    "components": list(component_names),
    "shapes": list(shape_names),
}
print(f"Loaded {n_scans:,} scans, {n_points} energies, {n_components} components")
print("Components:", component_names)

## 2. Split by case and normalize with training data only

The Anscombe transform reduces the count-dependent spread of Poisson noise. One global scale preserves differences in absolute count level. A log maximum-count feature makes count level explicit. Target center and widths are scaled for learning; height and area use `log1p` because they span a broad range. Missing shape-specific labels are replaced with zero **after** masks have been constructed.

In [ ]:
split = labels["split"]
indices = {name: np.flatnonzero(split == name) for name in ("train", "validation", "test")}
for name, idx in indices.items():
    assert len(idx) > 0, f"No {name} scans in the NeXus dataset"
case_sets = {name: set(labels["case_id"][idx].tolist()) for name, idx in indices.items()}
assert not (case_sets["train"] & case_sets["validation"])
assert not (case_sets["train"] & case_sets["test"])
assert not (case_sets["validation"] & case_sets["test"])
assert sum(len(idx) for idx in indices.values()) == n_scans

train_idx = indices["train"]
transformed = 2.0 * np.sqrt(np.maximum(spectra, 0.0) + 0.375)
input_scale = float(np.quantile(transformed[train_idx], 0.999))
count_scale = float(max(np.max(spectra[train_idx]), 1.0) * 1.1)
assert input_scale > 0

def make_input(counts):
    counts = np.asarray(counts, dtype=np.float32)
    if counts.ndim == 1:
        counts = counts[None, :]
    if counts.ndim != 2 or counts.shape[1] != n_points:
        raise ValueError(f"Expected an (N, {n_points}) count array")
    if not np.all(np.isfinite(counts)) or np.any(counts < 0):
        raise ValueError("Counts must be finite and nonnegative")
    anscombe = 2.0 * np.sqrt(counts + 0.375) / input_scale
    log_max = np.log1p(counts.max(axis=1, keepdims=True)) / np.log1p(count_scale)
    return np.concatenate((anscombe, log_max), axis=1).astype(np.float32)

x_all = make_input(spectra)
energy_min, energy_max = float(energy_ev[0]), float(energy_ev[-1])

def positive_scale(name, floor):
    train_values = labels[name][train_idx]
    finite_values = train_values[np.isfinite(train_values)]
    maximum = float(finite_values.max()) if finite_values.size else floor
    return float(max(maximum, floor) * 1.1)

scales = {
    "energy_min_ev": energy_min,
    "energy_span_ev": energy_max - energy_min,
    "fwhm_ev": positive_scale("peak_fwhm_ev", 0.1),
    "sigma_ev": positive_scale("peak_model_sigma_ev", 0.1),
    "lorentz_gamma_ev": positive_scale("peak_lorentz_gamma_ev", 0.1),
    "asymmetry_gamma": positive_scale("peak_asymmetry_gamma", 0.01),
    "height_counts": positive_scale("peak_height_counts", 1.0),
    "area_counts_ev": positive_scale("peak_area_counts_ev", 1.0),
    "input_anscombe": input_scale,
    "input_count": count_scale,
}

def finite32(values):
    return np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

targets = {
    "present": present.astype(np.float32),
    "shape_code": shape_code,
    "center": finite32((labels["peak_center_ev"] - energy_min) / scales["energy_span_ev"]),
    "fwhm": finite32(labels["peak_fwhm_ev"] / scales["fwhm_ev"]),
    "sigma": finite32(labels["peak_model_sigma_ev"] / scales["sigma_ev"]),
    "lorentz_gamma": finite32(labels["peak_lorentz_gamma_ev"] / scales["lorentz_gamma_ev"]),
    "pseudo_fraction": finite32(labels["peak_pseudo_voigt_fraction"]),
    "asymmetry_gamma": finite32(labels["peak_asymmetry_gamma"] / scales["asymmetry_gamma"]),
    "log_height": finite32(np.log1p(np.maximum(labels["peak_height_counts"], 0)) / np.log1p(scales["height_counts"])),
    "log_area": finite32(np.log1p(np.maximum(labels["peak_area_counts_ev"], 0)) / np.log1p(scales["area_counts_ev"])),
    "area_fraction": finite32(labels["peak_area_fraction"]),
}

class PeakDataset(Dataset):
    def __init__(self, row_indices):
        self.x = torch.from_numpy(x_all[row_indices])
        self.y = {key: torch.from_numpy(value[row_indices]) for key, value in targets.items()}

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], {key: value[index] for key, value in self.y.items()}

loaders = {
    name: DataLoader(PeakDataset(idx), batch_size=BATCH_SIZE, shuffle=(name == "train"), num_workers=0)
    for name, idx in indices.items()
}
positive_frequency = targets["present"][train_idx].mean(axis=0)
positive_weight = np.clip((1.0 - positive_frequency) / np.maximum(positive_frequency, 1e-6), 1.0, 8.0)
positive_weight = torch.tensor(positive_weight, dtype=torch.float32, device=DEVICE)
print({name: (len(idx), len(case_sets[name])) for name, idx in indices.items()})

## 3. Encoder, latent space, and peak-data decoder

The encoder maps the measured spectrum plus count-level feature to a 24-dimensional distribution. A sampled latent vector trains a decoder with one output slot per chemical component. Its heads output only peak attributes. At inference, the posterior mean gives a repeatable prediction. Profile choices obey the generator: ordinary components use Gaussian, Voigt, or pseudo-Voigt; a graphitic component may be Doniach Gaussian; the satellite uses pseudo-Voigt.

In [ ]:
# Present-shape class order corresponds to shape codes 1..4.
allowed_shapes = torch.zeros((n_components, len(shape_names) - 1), dtype=torch.bool)
allowed_shapes[:, [0, 1, 3]] = True
allowed_shapes[4, 2] = True
allowed_shapes[5, :] = False
allowed_shapes[5, 1] = True

class PeakParameterVAE(nn.Module):
    def __init__(self, input_dim, n_components, latent_dim, allowed_shapes):
        super().__init__()
        self.n_components = n_components
        self.register_buffer("allowed_shapes", allowed_shapes)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256), nn.SiLU(),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.SiLU(),
            nn.Linear(128, 64), nn.SiLU(),
        )
        self.to_mu = nn.Linear(64, latent_dim)
        self.to_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.LayerNorm(128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
        )
        self.presence_head = nn.Linear(128, n_components)
        self.shape_head = nn.Linear(128, n_components * allowed_shapes.shape[1])
        self.area_head = nn.Linear(128, n_components)
        self.peak_heads = nn.ModuleDict({
            name: nn.Linear(128, n_components) for name in (
                "center", "fwhm", "sigma", "lorentz_gamma", "pseudo_fraction",
                "asymmetry_gamma", "log_height",
            )
        })

    def forward(self, x, sample=True):
        encoded = self.encoder(x)
        mu = self.to_mu(encoded)
        logvar = self.to_logvar(encoded).clamp(-10.0, 6.0)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar) if sample else mu
        hidden = self.decoder(z)
        shape_logits = self.shape_head(hidden).reshape(-1, self.n_components, self.allowed_shapes.shape[1])
        shape_logits = shape_logits.masked_fill(~self.allowed_shapes[None, :, :], -1e4)
        output = {
            "mu": mu, "logvar": logvar,
            "presence_logits": self.presence_head(hidden),
            "shape_logits": shape_logits,
            "center": torch.sigmoid(self.peak_heads["center"](hidden)),
            "fwhm": F.softplus(self.peak_heads["fwhm"](hidden)),
            "sigma": F.softplus(self.peak_heads["sigma"](hidden)),
            "lorentz_gamma": F.softplus(self.peak_heads["lorentz_gamma"](hidden)),
            "pseudo_fraction": torch.sigmoid(self.peak_heads["pseudo_fraction"](hidden)),
            "asymmetry_gamma": F.softplus(self.peak_heads["asymmetry_gamma"](hidden)),
            "log_height": F.softplus(self.peak_heads["log_height"](hidden)),
            "log_area": F.softplus(self.area_head(hidden)),
        }
        return output

model = PeakParameterVAE(x_all.shape[1], n_components, LATENT_DIM, allowed_shapes).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Peak-data objective

Presence uses binary cross-entropy, shape uses cross-entropy on present peaks, and numerical parameters use masked Huber loss. Lorentzian gamma, pseudo-Voigt fraction, and Doniach asymmetry are learned only where applicable. The area-fraction term compares each scan's **relative component areas**; there is no intensity-vector or spectrum-reconstruction term. KL regularization keeps the latent distribution smooth.

In [ ]:
def masked_huber(prediction, target, mask):
    mask = mask.to(prediction.dtype)
    per_peak = F.smooth_l1_loss(prediction, target, reduction="none")
    return (per_peak * mask).sum() / mask.sum().clamp_min(1.0)

def peak_losses(output, target):
    present_mask = target["present"] > 0.5
    shape = target["shape_code"].long()
    shape_targets = (shape - 1).clamp_min(0)
    presence_loss = F.binary_cross_entropy_with_logits(
        output["presence_logits"], target["present"], pos_weight=positive_weight
    )
    shape_per_peak = F.cross_entropy(
        output["shape_logits"].reshape(-1, len(shape_names) - 1),
        shape_targets.reshape(-1), reduction="none"
    ).reshape_as(target["present"])
    shape_loss = (shape_per_peak * target["present"]).sum() / target["present"].sum().clamp_min(1.0)

    # Relative areas are derived from the same predicted absolute areas.
    log_area = output["log_area"] * np.log1p(scales["area_counts_ev"])
    area_counts = torch.expm1(log_area.clamp(max=30.0)).clamp_min(1e-6)
    area_logits = torch.log(area_counts).masked_fill(~present_mask, -1e4)
    fraction_loss = F.kl_div(
        F.log_softmax(area_logits, dim=1), target["area_fraction"], reduction="batchmean"
    )
    kl = -0.5 * torch.mean(torch.sum(
        1.0 + output["logvar"] - output["mu"].square() - output["logvar"].exp(), dim=1
    ))
    terms = {
        "presence": presence_loss,
        "shape": shape_loss,
        "center": masked_huber(output["center"], target["center"], present_mask),
        "fwhm": masked_huber(output["fwhm"], target["fwhm"], present_mask),
        "sigma": masked_huber(output["sigma"], target["sigma"], present_mask),
        "lorentz_gamma": masked_huber(output["lorentz_gamma"], target["lorentz_gamma"], present_mask & (shape == 1)),
        "pseudo_fraction": masked_huber(output["pseudo_fraction"], target["pseudo_fraction"], present_mask & (shape == 2)),
        "asymmetry_gamma": masked_huber(output["asymmetry_gamma"], target["asymmetry_gamma"], present_mask & (shape == 3)),
        "height": masked_huber(output["log_height"], target["log_height"], present_mask),
        "area": masked_huber(output["log_area"], target["log_area"], present_mask),
        "area_fraction": fraction_loss,
        "kl": kl,
    }
    terms["total"] = (
        terms["presence"] + 0.5 * terms["shape"] + terms["center"] +
        terms["fwhm"] + terms["sigma"] + terms["lorentz_gamma"] +
        terms["pseudo_fraction"] + terms["asymmetry_gamma"] +
        terms["height"] + terms["area"] + 0.5 * terms["area_fraction"] +
        BETA_KL * terms["kl"]
    )
    return terms

## 5. Train and choose a checkpoint on validation peak loss

Training samples the latent distribution. Validation decodes its mean. The held-out test split is used only after checkpoint selection. The checkpoint also stores the energy grid, label names, scales, and source-file manifest for later inference.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {part: {name: [] for name in (
    "total", "presence", "shape", "center", "fwhm", "sigma",
    "lorentz_gamma", "pseudo_fraction", "asymmetry_gamma",
    "height", "area", "area_fraction", "kl"
)} for part in ("train", "validation")}
best_val = float("inf")
best_epoch = 0
stale_epochs = 0

for epoch in range(1, EPOCHS + 1):
    for part in ("train", "validation"):
        training = part == "train"
        model.train(training)
        running = {key: 0.0 for key in history[part]}
        for batch_x, batch_y in loaders[part]:
            batch_x = batch_x.to(DEVICE)
            batch_y = {name: value.to(DEVICE) for name, value in batch_y.items()}
            with torch.set_grad_enabled(training):
                output = model(batch_x, sample=training)
                terms = peak_losses(output, batch_y)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    terms["total"].backward()
                    optimizer.step()
            for name in running:
                running[name] += float(terms[name].detach()) * len(batch_x)
        for name in running:
            history[part][name].append(running[name] / len(indices[part]))

    val = history["validation"]["total"][-1]
    if val < best_val - 1e-5:
        best_val = val
        best_epoch = epoch
        stale_epochs = 0
        torch.save({
            "state_dict": model.state_dict(), "epoch": epoch, "validation_loss": val,
            "input_dim": x_all.shape[1], "latent_dim": LATENT_DIM,
            "component_names": component_names, "shape_names": shape_names,
            "energy_ev": energy_ev, "scales": scales, "manifest": manifest,
            "presence_threshold": PRESENCE_THRESHOLD,
            "model_kind": "supervised_variational_peak_parameter_encoder_decoder",
        }, CHECKPOINT_PATH)
    else:
        stale_epochs += 1
    if epoch == 1 or epoch % 10 == 0 or stale_epochs == PATIENCE:
        print(f"Epoch {epoch:3d}: train={history['train']['total'][-1]:.4f}, "
              f"validation={val:.4f}, best={best_val:.4f} at {best_epoch}")
    if stale_epochs >= PATIENCE:
        break

with (OUTPUT_DIR / "training_history.json").open("w") as handle:
    json.dump(history, handle, indent=2)
print(f"Best checkpoint: {CHECKPOINT_PATH}")

## 6. Check peak-target learning

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for part in ("train", "validation"):
    epochs = np.arange(1, len(history[part]["total"]) + 1)
    axes[0].plot(epochs, history[part]["total"], label=part)
    axes[1].plot(epochs, history[part]["presence"], label=f"{part} presence")
    axes[1].plot(epochs, history[part]["shape"], label=f"{part} shape")
axes[0].set(title="Peak-data objective", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Presence and shape", xlabel="Epoch", ylabel="Loss")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "peak_training_curves.png", dpi=160)
plt.show()

## 7. Predict physical peak parameters

Each prediction is a candidate for lmfit optimization in the final cell. The threshold is adjustable; the reported FWHM is a separate geometric estimate, while `model_sigma_ev` is the profile parameter. On absent peaks, parameter fields are omitted. The satellite is suppressed if its aromatic/graphitic parent is absent.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
assert checkpoint["manifest"] == manifest, "The .nxs source changed after this checkpoint was trained"
assert tuple(checkpoint["component_names"]) == component_names
assert tuple(checkpoint["shape_names"]) == shape_names
model.load_state_dict(checkpoint["state_dict"])
model.eval()

def output_to_arrays(output, threshold=PRESENCE_THRESHOLD):
    raw = {name: value.detach().cpu().numpy() for name, value in output.items()
           if name not in ("mu", "logvar")}
    probability = 1.0 / (1.0 + np.exp(-raw["presence_logits"]))
    selected = probability >= threshold
    selected[:, 5] &= selected[:, 4]  # π–π* requires an aromatic/graphitic parent.
    shape = raw["shape_logits"].argmax(axis=2) + 1
    shape[~selected] = 0
    center_ev = energy_min + raw["center"] * scales["energy_span_ev"]
    fwhm_ev = raw["fwhm"] * scales["fwhm_ev"]
    sigma_ev = raw["sigma"] * scales["sigma_ev"]
    gamma_ev = raw["lorentz_gamma"] * scales["lorentz_gamma_ev"]
    asymmetry = raw["asymmetry_gamma"] * scales["asymmetry_gamma"]
    height = np.expm1(np.clip(raw["log_height"] * np.log1p(scales["height_counts"]), 0, 30))
    area = np.expm1(np.clip(raw["log_area"] * np.log1p(scales["area_counts_ev"]), 0, 30))
    selected_area = np.where(selected, area, 0.0)
    fraction = selected_area / np.maximum(selected_area.sum(axis=1, keepdims=True), 1e-12)
    return {
        "presence_probability": probability, "present": selected, "shape_code": shape,
        "center_ev": center_ev, "fwhm_ev": fwhm_ev, "model_sigma_ev": sigma_ev,
        "lorentz_gamma_ev": gamma_ev, "pseudo_voigt_fraction": raw["pseudo_fraction"],
        "asymmetry_gamma": asymmetry, "height_counts": height,
        "area_counts_ev": selected_area, "area_fraction": fraction,
    }

def predict_peaks(raw_counts, threshold=PRESENCE_THRESHOLD):
    """Predict one spectrum's lmfit candidate parameters on this common energy grid."""
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be between 0 and 1")
    x = torch.from_numpy(make_input(raw_counts)).to(DEVICE)
    if len(x) != 1:
        raise ValueError("predict_peaks accepts one spectrum at a time")
    with torch.no_grad():
        physical = output_to_arrays(model(x, sample=False), threshold)
    peaks = []
    for component, name in enumerate(component_names):
        candidate = {
            "component": name,
            "presence_probability": float(physical["presence_probability"][0, component]),
            "present": bool(physical["present"][0, component]),
        }
        if candidate["present"]:
            shape = int(physical["shape_code"][0, component])
            candidate.update({
                "profile": shape_names[shape],
                "center_ev": float(physical["center_ev"][0, component]),
                "fwhm_ev": float(physical["fwhm_ev"][0, component]),
                "model_sigma_ev": float(physical["model_sigma_ev"][0, component]),
                "height_counts": float(physical["height_counts"][0, component]),
                "area_counts_ev": float(physical["area_counts_ev"][0, component]),
                "area_fraction": float(physical["area_fraction"][0, component]),
            })
            if shape == 1:
                candidate["lorentz_gamma_ev"] = float(physical["lorentz_gamma_ev"][0, component])
            elif shape == 2:
                candidate["pseudo_voigt_fraction"] = float(physical["pseudo_voigt_fraction"][0, component])
            elif shape == 3:
                candidate["asymmetry_gamma"] = float(physical["asymmetry_gamma"][0, component])
        peaks.append(candidate)
    return peaks

print("Example held-out scan:")
print(json.dumps(predict_peaks(spectra[indices["test"][0]]), indent=2))

## 8. Evaluate held-out synthetic scans

Presence metrics use the inference threshold. Shape accuracy is measured for peaks the model and generator both mark present. Numerical errors are measured on correctly detected peaks; presence recall separately exposes missed peaks. Shape-specific errors use the matching true profile and a detected peak. These results measure recovery from this synthetic generator; measured XPS data still need an independent evaluation.

In [ ]:
outputs = []
with torch.no_grad():
    for batch_x, _ in loaders["test"]:
        outputs.append(output_to_arrays(model(batch_x.to(DEVICE), sample=False)))
pred = {name: np.concatenate([part[name] for part in outputs], axis=0)
        for name in outputs[0]}
true_present = present[indices["test"]]
pred_present = pred["present"]
true_shape = shape_code[indices["test"]]
matched = true_present & pred_present

def mae(predicted, actual, mask):
    return float(np.mean(np.abs(predicted[mask] - actual[mask]))) if np.any(mask) else None

tp = int(np.sum(pred_present & true_present))
fp = int(np.sum(pred_present & ~true_present))
fn = int(np.sum(~pred_present & true_present))
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
metrics = {
    "scan_count": int(len(indices["test"])),
    "true_peaks": int(true_present.sum()),
    "presence_precision": precision,
    "presence_recall": recall,
    "presence_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
    "shape_accuracy_on_detected_true_peaks": float(np.mean(pred["shape_code"][matched] == true_shape[matched])) if np.any(matched) else None,
    "center_mae_ev_on_detected_peaks": mae(pred["center_ev"], labels["peak_center_ev"][indices["test"]], matched),
    "fwhm_mae_ev_on_detected_peaks": mae(pred["fwhm_ev"], labels["peak_fwhm_ev"][indices["test"]], matched),
    "model_sigma_mae_ev_on_detected_peaks": mae(pred["model_sigma_ev"], labels["peak_model_sigma_ev"][indices["test"]], matched),
    "lorentz_gamma_mae_ev_on_detected_voigt": mae(pred["lorentz_gamma_ev"], labels["peak_lorentz_gamma_ev"][indices["test"]], matched & (true_shape == 1)),
    "pseudo_fraction_mae_on_detected_pseudo_voigt": mae(pred["pseudo_voigt_fraction"], labels["peak_pseudo_voigt_fraction"][indices["test"]], matched & (true_shape == 2)),
    "asymmetry_gamma_mae_on_detected_doniach": mae(pred["asymmetry_gamma"], labels["peak_asymmetry_gamma"][indices["test"]], matched & (true_shape == 3)),
    "height_mae_counts_on_detected_peaks": mae(pred["height_counts"], labels["peak_height_counts"][indices["test"]], matched),
    "area_mae_counts_ev_on_detected_peaks": mae(pred["area_counts_ev"], labels["peak_area_counts_ev"][indices["test"]], matched),
    "area_fraction_mae_on_detected_peaks": mae(pred["area_fraction"], labels["peak_area_fraction"][indices["test"]], matched),
}
print(json.dumps(metrics, indent=2))

fieldnames = [
    "scan_index", "case_id", "family", "background_kind", "component",
    "true_present", "predicted_present", "presence_probability", "true_shape", "predicted_shape",
    "true_center_ev", "predicted_center_ev", "true_fwhm_ev", "predicted_fwhm_ev",
    "true_model_sigma_ev", "predicted_model_sigma_ev", "true_height_counts", "predicted_height_counts",
    "true_area_counts_ev", "predicted_area_counts_ev", "true_area_fraction", "predicted_area_fraction",
    "true_lorentz_gamma_ev", "predicted_lorentz_gamma_ev",
    "true_pseudo_voigt_fraction", "predicted_pseudo_voigt_fraction",
    "true_asymmetry_gamma", "predicted_asymmetry_gamma",
]
with (OUTPUT_DIR / "test_peak_predictions.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for local, scan_index in enumerate(indices["test"]):
        for component, component_name in enumerate(component_names):
            def value(name):
                result = labels[name][scan_index, component]
                return "" if not np.isfinite(result) else float(result)
            writer.writerow({
                "scan_index": int(scan_index), "case_id": int(labels["case_id"][scan_index]),
                "family": labels["family"][scan_index],
                "background_kind": labels["background_kind"][scan_index],
                "component": component_name,
                "true_present": bool(true_present[local, component]),
                "predicted_present": bool(pred_present[local, component]),
                "presence_probability": float(pred["presence_probability"][local, component]),
                "true_shape": shape_names[int(true_shape[local, component])],
                "predicted_shape": shape_names[int(pred["shape_code"][local, component])],
                "true_center_ev": value("peak_center_ev"),
                "predicted_center_ev": float(pred["center_ev"][local, component]) if pred_present[local, component] else "",
                "true_fwhm_ev": value("peak_fwhm_ev"),
                "predicted_fwhm_ev": float(pred["fwhm_ev"][local, component]) if pred_present[local, component] else "",
                "true_model_sigma_ev": value("peak_model_sigma_ev"),
                "predicted_model_sigma_ev": float(pred["model_sigma_ev"][local, component]) if pred_present[local, component] else "",
                "true_height_counts": value("peak_height_counts"),
                "predicted_height_counts": float(pred["height_counts"][local, component]) if pred_present[local, component] else "",
                "true_area_counts_ev": value("peak_area_counts_ev"),
                "predicted_area_counts_ev": float(pred["area_counts_ev"][local, component]) if pred_present[local, component] else "",
                "true_area_fraction": value("peak_area_fraction"),
                "predicted_area_fraction": float(pred["area_fraction"][local, component]) if pred_present[local, component] else "",
                "true_lorentz_gamma_ev": value("peak_lorentz_gamma_ev"),
                "predicted_lorentz_gamma_ev": float(pred["lorentz_gamma_ev"][local, component]) if pred["shape_code"][local, component] == 1 else "",
                "true_pseudo_voigt_fraction": value("peak_pseudo_voigt_fraction"),
                "predicted_pseudo_voigt_fraction": float(pred["pseudo_voigt_fraction"][local, component]) if pred["shape_code"][local, component] == 2 else "",
                "true_asymmetry_gamma": value("peak_asymmetry_gamma"),
                "predicted_asymmetry_gamma": float(pred["asymmetry_gamma"][local, component]) if pred["shape_code"][local, component] == 3 else "",
            })

## 9. Inspect a few predictions and save the summary

The spectrum below is shown only as input context; the model has no output spectrum. Vertical markers compare true and predicted component centers.

In [ ]:
example_count = min(4, len(indices["test"]))
example_local = np.random.default_rng(SEED + 100).choice(len(indices["test"]), example_count, replace=False)
fig, axes = plt.subplots(example_count, 1, figsize=(11, 2.8 * example_count), squeeze=False)
for ax, local in zip(axes[:, 0], example_local):
    scan_index = indices["test"][local]
    ax.plot(energy_ev, spectra[scan_index], color="0.25", lw=1, label="Noisy input")
    for component in range(n_components):
        if true_present[local, component]:
            ax.axvline(labels["peak_center_ev"][scan_index, component], color="tab:green", alpha=0.35)
        if pred_present[local, component]:
            ax.axvline(pred["center_ev"][local, component], color="tab:orange", alpha=0.7, ls="--")
    ax.set(title=f"Case {labels['case_id'][scan_index]} · {labels['family'][scan_index]} · {labels['background_kind'][scan_index]}",
           xlabel="Binding energy (eV)", ylabel="Counts")
axes[0, 0].plot([], [], color="tab:green", label="True center")
axes[0, 0].plot([], [], color="tab:orange", ls="--", label="Predicted center")
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_peak_examples.png", dpi=160)
plt.show()

summary = {
    "model_kind": "supervised_variational_peak_parameter_encoder_decoder",
    "source": manifest,
    "best_epoch": best_epoch,
    "best_validation_peak_loss": best_val,
    "presence_threshold": PRESENCE_THRESHOLD,
    "latent_dim": LATENT_DIM,
    "test_metrics": metrics,
    "checkpoint": str(CHECKPOINT_PATH),
    "prediction_csv": str(OUTPUT_DIR / "test_peak_predictions.csv"),
}
with (OUTPUT_DIR / "test_summary.json").open("w") as handle:
    json.dump(summary, handle, indent=2)
print(f"Saved outputs in {OUTPUT_DIR}")

In [ ]:
# 10. Fit one held-out spectrum with lmfit, initialized only by VAE peak predictions.
# Change FIT_SCAN_INDEX to another index in indices["test"] to inspect another scan.
from lmfit import Parameters, minimize, fit_report
from lmfit.models import DoniachModel, GaussianModel, PseudoVoigtModel, VoigtModel
from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.ndimage import gaussian_filter1d
from scipy.signal import fftconvolve

FIT_SCAN_INDEX = int(indices["test"][0])
FIT_PRESENCE_THRESHOLD = PRESENCE_THRESHOLD
assert FIT_SCAN_INDEX in set(indices["test"].tolist()), "Choose a held-out test scan"
x_fit = np.asarray(energy_ev, dtype=np.float64)
y_fit = np.asarray(spectra[FIT_SCAN_INDEX], dtype=np.float64)
step_ev = float(np.median(np.diff(x_fit)))
fit_guesses = predict_peaks(y_fit, threshold=FIT_PRESENCE_THRESHOLD)
active_guesses = [guess for guess in fit_guesses if guess["present"]]
if not active_guesses:
    raise ValueError("The VAE predicted no peaks. Inspect the scan or lower FIT_PRESENCE_THRESHOLD.")

model_types = {
    "gaussian": GaussianModel,
    "voigt": VoigtModel,
    "pseudo_voigt": PseudoVoigtModel,
    "doniach_gaussian": DoniachModel,
}
component_models = []
peak_params = Parameters()
needs_doniach_broadening = any(g["profile"] == "doniach_gaussian" for g in active_guesses)
if needs_doniach_broadening:
    # The generator convolves Doniach peaks with an instrument Gaussian. The VAE
    # predicts peak parameters but not instrument FWHM, so lmfit estimates it here.
    peak_params.add("instrument_fwhm_ev", value=0.55, min=0.20, max=1.20)

for slot, guess in enumerate(active_guesses):
    prefix = f"p{slot}_"
    profile = guess["profile"]
    model = model_types[profile](prefix=prefix)
    center = float(np.clip(guess["center_ev"], x_fit[0] + 0.1, x_fit[-1] - 0.1))
    sigma = float(np.clip(guess["model_sigma_ev"], 0.05, 3.0))
    local = model.make_params(amplitude=1.0, center=center, sigma=sigma)
    if profile == "voigt":
        # lmfit normally ties Voigt gamma to sigma; the VAE predicts gamma separately.
        local[prefix + "gamma"].set(
            value=float(np.clip(guess["lorentz_gamma_ev"], 0.01, 0.8)),
            min=0.005, max=1.0, expr="",
        )
    elif profile == "pseudo_voigt":
        local[prefix + "fraction"].set(
            value=float(np.clip(guess["pseudo_voigt_fraction"], 0.01, 0.99)),
            min=0.0, max=1.0,
        )
    elif profile == "doniach_gaussian":
        local[prefix + "gamma"].set(
            value=float(np.clip(guess["asymmetry_gamma"], 0.01, 0.3)),
            min=0.001, max=0.35,
        )

    # lmfit amplitude has different meanings across profile families. Normalize
    # an amplitude-one profile on this finite scan to initialize from VAE area.
    unit = np.asarray(model.eval(params=local, x=x_fit), dtype=np.float64)
    if profile == "doniach_gaussian":
        unit = gaussian_filter1d(
            unit, 0.55 / (2.0 * np.sqrt(2.0 * np.log(2.0)) * step_ev), mode="constant"
        )
    unit_area = float(trapezoid(unit, x_fit))
    if not np.isfinite(unit_area) or unit_area <= 0:
        raise ValueError(f"Invalid initial profile for {guess['component']}")
    initial_amplitude = max(float(guess["area_counts_ev"]) / unit_area, 1e-6)
    local[prefix + "amplitude"].set(
        value=initial_amplitude, min=0.0, max=max(initial_amplitude * 30.0, 1e3)
    )
    local[prefix + "center"].set(
        value=center, min=max(float(x_fit[0]), center - 0.9),
        max=min(float(x_fit[-1]), center + 0.9),
    )
    local[prefix + "sigma"].set(value=sigma, min=0.04, max=max(3.0, sigma * 2.0))
    peak_params.update(local)
    component_models.append({
        "component": guess["component"], "profile": profile,
        "prefix": prefix, "model": model,
    })

edge_count = max(5, min(15, len(y_fit) // 20))
edge_low = max(float(np.median(y_fit[:edge_count])), 0.0)
edge_high = max(float(np.median(y_fit[-edge_count:])), 0.0)


def evaluate_fit(params, background_kind):
    """Return total, individual peak curves, and background on the common grid."""
    parts = {}
    for item in component_models:
        curve = np.asarray(item["model"].eval(params=params, x=x_fit), dtype=np.float64)
        if item["profile"] == "doniach_gaussian":
            sigma_points = (
                params["instrument_fwhm_ev"].value /
                (2.0 * np.sqrt(2.0 * np.log(2.0)) * step_ev)
            )
            curve = gaussian_filter1d(curve, sigma_points, mode="constant")
        parts[item["component"]] = curve
    peak_sum = np.sum(list(parts.values()), axis=0)
    position = (x_fit - x_fit[0]) / (x_fit[-1] - x_fit[0])
    background = params["background_low"].value + (
        params["background_high"].value - params["background_low"].value
    ) * position
    if background_kind == "shirley":
        cumulative = cumulative_trapezoid(peak_sum, x_fit, initial=0.0)
        background = background + (
            params["shirley_fraction"].value * peak_sum.max() *
            cumulative / max(float(cumulative[-1]), 1e-12)
        )
    elif background_kind == "tougaard":
        loss_ev = x_fit - x_fit[0]
        b_ev2 = params["tougaard_B_eV2"].value
        kernel = b_ev2 * loss_ev / (1643.0 + loss_ev**2)**2
        loss_signal = np.zeros_like(peak_sum)
        for _ in range(20):
            updated = fftconvolve(peak_sum + loss_signal, kernel, mode="full")[:len(x_fit)] * step_ev
            updated = np.maximum(updated, 0.0)
            if np.max(np.abs(updated - loss_signal)) < 1e-8 * max(float(peak_sum.max()), 1.0):
                loss_signal = updated
                break
            loss_signal = updated
        background = background + loss_signal
    return peak_sum + background, parts, background


def weighted_residual(params, background_kind):
    fitted, _, _ = evaluate_fit(params, background_kind)
    # Native Poisson scans are interpolated, so these are approximate weights.
    return (fitted - y_fit) / np.sqrt(np.maximum(y_fit, 1.0))


# Try all three forward-background families without using the hidden true label.
fit_candidates = []
for background_kind in ("linear", "shirley", "tougaard"):
    params = peak_params.copy()
    baseline_ceiling = max(float(y_fit.max()) * 2.0, 1.0)
    params.add("background_low", value=edge_low, min=0.0, max=baseline_ceiling)
    params.add("background_high", value=edge_high, min=0.0, max=baseline_ceiling)
    if background_kind == "shirley":
        params.add("shirley_fraction", value=0.12, min=0.0, max=0.5)
    elif background_kind == "tougaard":
        params.add("tougaard_B_eV2", value=2866.0, min=1000.0, max=5000.0)
    initial_curve, _, _ = evaluate_fit(params, background_kind)
    result = minimize(
        weighted_residual, params, args=(background_kind,),
        method="least_squares", max_nfev=1500,
    )
    if np.isfinite(result.aic):
        fit_candidates.append((float(result.aic), background_kind, result, initial_curve))
if not fit_candidates:
    raise RuntimeError("lmfit did not return a finite fit for any background family")

converged_candidates = [candidate for candidate in fit_candidates if candidate[2].success]
if not converged_candidates:
    print("Warning: no candidate converged within max_nfev; inspect the fit report and residuals.")
_, selected_background, fit_result, initial_curve = min(
    converged_candidates or fit_candidates, key=lambda item: item[0]
)
fitted_curve, fitted_parts, fitted_background = evaluate_fit(fit_result.params, selected_background)
fit_error = y_fit - fitted_curve
print(f"Case {labels['case_id'][FIT_SCAN_INDEX]}, scan {FIT_SCAN_INDEX}")
print(f"Selected background: {selected_background}; AIC: {fit_result.aic:.2f}; success: {fit_result.success}")
print(fit_report(fit_result))

# Save a usable fitted spectrum and the component curves.
fit_csv_path = OUTPUT_DIR / f"lmfit_scan_{FIT_SCAN_INDEX:05d}.csv"
curve_columns = {
    "binding_energy_ev": x_fit,
    "observed_counts": y_fit,
    "vae_initialized_counts": initial_curve,
    "lmfit_fitted_counts": fitted_curve,
    "fitted_background_counts": fitted_background,
    "residual_counts": fit_error,
}
for component, curve in fitted_parts.items():
    curve_columns[f"{component}_counts"] = curve
with fit_csv_path.open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(curve_columns)
    writer.writerows(zip(*curve_columns.values()))

def numerical_fwhm(curve):
    maximum = int(np.argmax(curve))
    half_height = float(curve[maximum]) / 2.0
    left = np.flatnonzero(curve[:maximum] <= half_height)
    right = np.flatnonzero(curve[maximum + 1:] <= half_height)
    if not len(left) or not len(right):
        return None
    lo, hi = int(left[-1]), int(maximum + 1 + right[0])
    x_left = float(np.interp(half_height, curve[lo:lo + 2], x_fit[lo:lo + 2]))
    x_right = float(np.interp(
        half_height, curve[hi - 1:hi + 1][::-1], x_fit[hi - 1:hi + 1][::-1]
    ))
    return x_right - x_left


fitted_peak_data = []
for item in component_models:
    prefix = item["prefix"]
    curve = fitted_parts[item["component"]]
    peak = {
        "component": item["component"], "profile": item["profile"],
        "center_ev": float(fit_result.params[prefix + "center"].value),
        "model_sigma_ev": float(fit_result.params[prefix + "sigma"].value),
        "lmfit_amplitude": float(fit_result.params[prefix + "amplitude"].value),
        "height_counts": float(curve.max()),
        "area_counts_ev": float(trapezoid(curve, x_fit)),
        "fwhm_ev": numerical_fwhm(curve),
    }
    if item["profile"] == "voigt":
        peak["lorentz_gamma_ev"] = float(fit_result.params[prefix + "gamma"].value)
    elif item["profile"] == "doniach_gaussian":
        peak["asymmetry_gamma"] = float(fit_result.params[prefix + "gamma"].value)
        peak["instrument_fwhm_ev"] = float(fit_result.params["instrument_fwhm_ev"].value)
    elif item["profile"] == "pseudo_voigt":
        peak["pseudo_voigt_fraction"] = float(fit_result.params[prefix + "fraction"].value)
    fitted_peak_data.append(peak)

area_sum = sum(peak["area_counts_ev"] for peak in fitted_peak_data)
for peak in fitted_peak_data:
    peak["area_fraction"] = peak["area_counts_ev"] / area_sum if area_sum > 0 else 0.0

fit_summary_path = OUTPUT_DIR / f"lmfit_scan_{FIT_SCAN_INDEX:05d}.json"
with fit_summary_path.open("w") as handle:
    json.dump({
        "scan_index": FIT_SCAN_INDEX,
        "case_id": int(labels["case_id"][FIT_SCAN_INDEX]),
        "selected_background": selected_background,
        "aic": float(fit_result.aic),
        "success": bool(fit_result.success),
        "vae_initial_peaks": active_guesses,
        "lmfit_peaks": fitted_peak_data,
    }, handle, indent=2)

fig, (ax, residual_ax) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
ax.plot(x_fit, y_fit, color="0.25", lw=1, label="Observed scan")
ax.plot(x_fit, initial_curve, color="tab:blue", ls="--", lw=1.3, label="VAE initialized model")
ax.plot(x_fit, fitted_curve, color="tab:red", lw=1.6, label="lmfit result")
ax.plot(x_fit, fitted_background, color="0.55", ls=":", label=f"{selected_background} background")
for component, curve in fitted_parts.items():
    ax.plot(x_fit, curve + fitted_background, lw=0.9, alpha=0.65, label=component)
ax.set(ylabel="Counts", title=f"VAE initialized lmfit · test scan {FIT_SCAN_INDEX}")
ax.legend(ncol=3, fontsize=8)
residual_ax.plot(x_fit, fit_error, color="tab:purple", lw=1)
residual_ax.axhline(0, color="0.5", lw=0.8)
residual_ax.set(xlabel="Binding energy (eV)", ylabel="Residual")
fig.tight_layout()
fit_plot_path = OUTPUT_DIR / f"lmfit_scan_{FIT_SCAN_INDEX:05d}.png"
fig.savefig(fit_plot_path, dpi=180)
plt.show()
print(f"Saved {fit_csv_path}, {fit_summary_path}, and {fit_plot_path}")